# Phase 8: Analyse the results

This notebook reads the 12 finished training runs from your Drive and turns them into the tables and figures for your paper. **No training happens here, so it doesn't need a GPU.** Runtime → Change runtime type → **CPU** saves your free GPU time.

Everything it makes is saved to `results/analysis/` in your Drive.

### Step 1: Load the results
Reads `results.csv` (one row per training run) and the class names.

**Expected output:** "12 runs | 15 classes" and the table of all runs.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/plant_health_capstone/results'
OUT_DIR = f'{RESULTS_DIR}/analysis'
os.makedirs(OUT_DIR, exist_ok=True)

results = pd.read_csv(f'{RESULTS_DIR}/results.csv')
split = pd.read_csv(f'{RESULTS_DIR}/data_split.csv')
class_names = split.drop_duplicates('label').sort_values('label')['file'].str.split('/').str[0].tolist()
short_names = [n.replace('Pepper__bell___', 'Pepper ').replace('Tomato__Tomato_', 'Tomato ')
                .replace('___', ' ').replace('__', ' ').replace('_', ' ') for n in class_names]

CONDITIONS = [('mobilenet', 'rgb'), ('mobilenet', 'hsv'), ('scratch', 'rgb'), ('scratch', 'hsv')]
LABELS = {('mobilenet', 'rgb'): 'MobileNet RGB', ('mobilenet', 'hsv'): 'MobileNet HSV',
          ('scratch', 'rgb'): 'Scratch RGB', ('scratch', 'hsv'): 'Scratch HSV'}

def runs_of(arch, colour):
    return results[(results.arch == arch) & (results.colour_space == colour)]

print(len(results), 'runs |', len(class_names), 'classes')
results

### Step 2: Table 1, the main results
Average ± spread over the 3 runs of each model, in %. The **lighting drop** is how many points accuracy falls from the normal test photos to the relit ones. Smaller means more robust to lighting.

In [ ]:
results['lighting_drop'] = results['test_acc'] - results['relit_test_acc']
metrics = {'test_acc': 'Test accuracy', 'test_macro_f1': 'Test macro F1',
           'relit_test_acc': 'Relit accuracy', 'relit_macro_f1': 'Relit macro F1',
           'lighting_drop': 'Lighting drop'}
rows = []
for c in CONDITIONS:
    sub = runs_of(*c)
    row = {'Model': LABELS[c], 'Runs': len(sub)}
    for m, name in metrics.items():
        row[name] = f'{100 * sub[m].mean():.1f} ± {100 * sub[m].std():.1f}'
    rows.append(row)
table1 = pd.DataFrame(rows)
table1.to_csv(f'{OUT_DIR}/table1_main_results.csv', index=False)
table1

### Step 3: Table 2, is the HSV vs RGB difference real?
For each model design, this compares the 3 HSV runs with the 3 RGB runs using a **Welch t-test**. The **p-value** is roughly the chance of seeing a gap this big if colour space made no real difference. Below 0.05 is the usual bar for "probably real".

With only 3 runs per side these tests are rough, so say that in your paper.

In [ ]:
rows = []
for arch in ['mobilenet', 'scratch']:
    for m in ['test_acc', 'test_macro_f1', 'relit_test_acc', 'relit_macro_f1', 'lighting_drop']:
        rgb = runs_of(arch, 'rgb')[m]
        hsv = runs_of(arch, 'hsv')[m]
        t, p = stats.ttest_ind(hsv, rgb, equal_var=False)
        rows.append({'Model': arch, 'Metric': metrics[m],
                     'RGB': round(100 * rgb.mean(), 1), 'HSV': round(100 * hsv.mean(), 1),
                     'HSV minus RGB': round(100 * (hsv.mean() - rgb.mean()), 1),
                     'p-value': round(p, 4), 'Below 0.05?': 'yes' if p < 0.05 else 'no'})
table2 = pd.DataFrame(rows)
table2.to_csv(f'{OUT_DIR}/table2_hsv_vs_rgb_tests.csv', index=False)
table2

### Step 4: Figure 1, accuracy at a glance
Bars are the average of the 3 runs, black dots are the individual runs, and the error bars show the spread. This is the main figure for your results section.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(CONDITIONS))
for metric, name, offset in [('test_acc', 'Normal test photos', -0.2), ('relit_test_acc', 'Relit test photos', 0.2)]:
    means, spreads = [], []
    for j, c in enumerate(CONDITIONS):
        vals = 100 * runs_of(*c)[metric]
        means.append(vals.mean())
        spreads.append(vals.std())
        ax.scatter([j + offset] * len(vals), vals, color='black', s=12, zorder=3)
    ax.bar(x + offset, means, width=0.4, yerr=spreads, capsize=4, label=name)
ax.set_xticks(x)
ax.set_xticklabels([LABELS[c] for c in CONDITIONS])
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(0, 112)
ax.set_yticks(range(0, 101, 20))
ax.set_title('Accuracy by model design and colour space')
ax.legend(loc='upper center', ncol=2, frameon=False)
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/fig1_accuracy.png', dpi=200)
plt.show()

### Step 5: Figure 2, which diseases each model gets right
**F1** is a per-class score from 0 to 1 that balances "when it says this disease, is it right?" with "does it find all the leaves with this disease?". Each cell is the average over 3 runs.

Look for rows where one colour space is clearly greener than the other. That tells you *which* diseases HSV helps or hurts, which is the most interesting part of your discussion section.

In [ ]:
def per_class_f1(cm):
    tp = np.diag(cm).astype(float)
    predicted, actual = cm.sum(axis=0), cm.sum(axis=1)
    precision = np.divide(tp, predicted, out=np.zeros_like(tp), where=predicted > 0)
    recall = np.divide(tp, actual, out=np.zeros_like(tp), where=actual > 0)
    total = precision + recall
    return np.divide(2 * precision * recall, total, out=np.zeros_like(tp), where=total > 0)

def confusion_matrices(arch, colour, which):
    return [np.load(f'{RESULTS_DIR}/{arch}_{colour}_seed{s}/confusion_{which}.npy') for s in runs_of(arch, colour).seed]

f1 = {}
for which in ['test', 'relit']:
    f1[which] = pd.DataFrame({LABELS[c]: np.mean([per_class_f1(cm) for cm in confusion_matrices(*c, which)], axis=0)
                              for c in CONDITIONS}, index=short_names)
    f1[which].round(3).to_csv(f'{OUT_DIR}/per_class_f1_{which}.csv')

fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
for ax, which, title in zip(axes, ['test', 'relit'], ['Normal test photos', 'Relit test photos']):
    values = f1[which].values
    im = ax.imshow(values, cmap='viridis', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(values.shape[1]))
    ax.set_xticklabels(f1[which].columns, rotation=30, ha='right')
    ax.set_yticks(range(values.shape[0]))
    ax.set_yticklabels(f1[which].index)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            ax.text(j, i, f'{values[i, j]:.2f}', ha='center', va='center', fontsize=8,
                    color='white' if values[i, j] < 0.6 else 'black')
    ax.set_title(f'Per-class F1: {title}')
fig.colorbar(im, ax=axes, shrink=0.6)
fig.savefig(f'{OUT_DIR}/fig2_per_class_f1.png', dpi=200, bbox_inches='tight')
plt.show()

test_counts = confusion_matrices('scratch', 'rgb', 'test')[0].sum(axis=1)
print('Test photos per class:', dict(zip(short_names, test_counts.tolist())))

### Step 6: Figure 3, where HSV helps the scratch model
For the from-scratch design (the fair test of your hypothesis), this shows HSV's F1 minus RGB's F1 for each class. Bars to the right mean HSV did better on that disease.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
for ax, which, title in zip(axes, ['test', 'relit'], ['Normal test photos', 'Relit test photos']):
    diff = (f1[which]['Scratch HSV'] - f1[which]['Scratch RGB']).sort_values()
    ax.barh(diff.index, diff.values, color=['tab:red' if v < 0 else 'tab:green' for v in diff.values])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Scratch model, HSV minus RGB F1: {title}')
    ax.set_xlabel('F1 difference')
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/fig3_scratch_hsv_gain_by_class.png', dpi=200)
plt.show()

### Step 7: Table 3, the most common mistakes
For each model, the 5 most frequent mix-ups on the normal test photos, added up across its 3 runs. For example, "early blight predicted as target spot" means a leaf that really had early blight was labelled target spot.

In [ ]:
def top_mistakes(arch, colour, which='test', n=5):
    cm = np.sum(confusion_matrices(arch, colour, which), axis=0)
    rows = []
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            if i != j and cm[i, j] > 0:
                rows.append({'Model': LABELS[(arch, colour)], 'True class': short_names[i],
                             'Predicted as': short_names[j], 'Photos (3 runs)': int(cm[i, j]),
                             '% of that class': round(100 * cm[i, j] / cm[i].sum(), 1)})
    return pd.DataFrame(rows).sort_values('Photos (3 runs)', ascending=False).head(n)

table3 = pd.concat([top_mistakes(*c) for c in CONDITIONS], ignore_index=True)
table3.to_csv(f'{OUT_DIR}/table3_top_mistakes.csv', index=False)
table3

### Step 8: Table 4 and Figure 4, how stable was training?
The scratch models' validation accuracy jumped around from epoch to epoch, and each run was scored at its **final** epoch, whatever it happened to be. This table puts a number on that, so you can report it honestly as a limitation:
- **Final vs best:** the validation accuracy the run ended on, compared with its best epoch.
- **Swing:** the gap between the highest and lowest validation accuracy over the last 5 epochs.

In [ ]:
histories = {}
rows = []
for _, r in results.iterrows():
    h = pd.read_csv(f'{RESULTS_DIR}/{r.arch}_{r.colour_space}_seed{r.seed}/history.csv')
    histories[(r.arch, r.colour_space, r.seed)] = h
    last5 = h['val_accuracy'].tail(5)
    rows.append({'Model': LABELS[(r.arch, r.colour_space)], 'Seed': r.seed,
                 'Final val acc': round(100 * h['val_accuracy'].iloc[-1], 1),
                 'Best val acc': round(100 * h['val_accuracy'].max(), 1),
                 'Best epoch': int(h['val_accuracy'].idxmax()) + 1,
                 'Swing, last 5 epochs': round(100 * (last5.max() - last5.min()), 1)})
table4 = pd.DataFrame(rows)
table4.to_csv(f'{OUT_DIR}/table4_training_stability.csv', index=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for ax, c in zip(axes.flat, CONDITIONS):
    for s in runs_of(*c).seed:
        h = histories[(c[0], c[1], s)]
        ax.plot(range(1, len(h) + 1), 100 * h['val_accuracy'], marker='o', markersize=3, label=f'seed {s}')
    ax.set_title(LABELS[c])
    ax.set_xlabel('epoch')
    ax.set_ylabel('validation accuracy (%)')
    ax.legend()
fig.suptitle('Validation accuracy during training')
fig.tight_layout()
fig.savefig(f'{OUT_DIR}/fig4_validation_curves.png', dpi=200)
plt.show()
table4

### Step 9: What was saved
All tables (CSV) and figures (PNG) are in `results/analysis/`. You can open the CSVs in Google Sheets and drop the PNGs straight into your paper.

In [ ]:
for f in sorted(os.listdir(OUT_DIR)):
    print(f)